In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


def seed_all(seed: int = 20) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_all(20)




In [2]:
train_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv"
test_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv"
sample_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"

train = pd.read_csv(train_path)
raw_test = pd.read_csv(test_path)  # contains only the baseline row per patient
submission_template = pd.read_csv(sample_path)

baseline_train = train[train["Weeks"] == 0][["Patient", "FVC"]].rename(
    columns={"FVC": "FVC_base"}
)
train = train.merge(baseline_train, on="Patient", how="left")
train["FVC_base"].fillna(train["FVC_base"].median(), inplace=True)




/tmp/ipykernel_11/981453938.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train["FVC_base"].fillna(train["FVC_base"].median(), inplace=True)


In [3]:
cat_cols = ["Sex", "SmokingStatus"]
encoders = {c: LabelEncoder() for c in cat_cols}
for col in cat_cols:
    encoders[col].fit(pd.concat([train[col], raw_test[col]], axis=0))
    train[col] = encoders[col].transform(train[col])
    raw_test[col] = encoders[col].transform(raw_test[col])




In [4]:
# add a simple non‑linear feature to help the model capture curvature in time
train["Weeks_sq"] = train["Weeks"] ** 2
raw_test["Weeks_sq"] = raw_test["Weeks"] ** 2

SELECTED_COLUMNS = [
    "Weeks",
    "Weeks_sq",  # new feature
    "Percent",
    "Age",
    "Sex",
    "SmokingStatus",
    "FVC_base",
]

X_train = train[SELECTED_COLUMNS]
y_train = train["FVC"].astype(np.float32)

NFOLD = 5
kf = KFold(n_splits=NFOLD, shuffle=True, random_state=20)

oof_pred = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = GradientBoostingRegressor(
        n_estimators=800, learning_rate=0.05, max_depth=3, random_state=20
    )
    model.fit(X_tr, y_tr)
    oof_pred[val_idx] = model.predict(X_val)

sigma_median = np.median(np.abs(train["FVC"] - oof_pred))
sigma_mean = np.mean(np.abs(train["FVC"] - oof_pred))  # kept for reference only


def competition_metric(y_true, y_pred, sigma):
    sigma_clipped = np.maximum(sigma, 70.0)
    delta = np.minimum(np.abs(y_true - y_pred), 1000.0)
    return -np.sqrt(2) * delta / sigma_clipped - np.log(np.sqrt(2) * sigma_clipped)


factors = np.arange(0.5, 30.05, 0.05)  # explore a larger range
best_score = -np.inf
best_factor = 1.0
for f in factors:
    sigma_f = sigma_median * f
    scores = competition_metric(train["FVC"].values, oof_pred, sigma_f)
    mean_score = scores.mean()
    if mean_score > best_score:
        best_score = mean_score
        best_factor = f




In [5]:
sub = submission_template.copy()
sub["Patient"] = sub["Patient_Week"].str.extract(r"^(.*)_\d+$")
sub["Weeks"] = sub["Patient_Week"].str.split("_").str[-1].astype(int)

sub = sub.merge(raw_test, on="Patient", how="left", suffixes=("", "_base"))

if "FVC" in sub.columns:
    sub["FVC_base"] = sub["FVC"]
    sub.drop(columns=["FVC"], inplace=True)

if "Weeks_base" in sub.columns:
    sub.drop(columns=["Weeks_base"], inplace=True)

X_sub = sub[SELECTED_COLUMNS].copy()
X_sub = X_sub.reindex(columns=SELECTED_COLUMNS)  # guarantees exact column order
X_sub.fillna(0, inplace=True)  # safety net; should not be triggered




In [6]:
full_model = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=3, random_state=20
)
full_model.fit(X_train, y_train)

sub["FVC"] = full_model.predict(X_sub)

scaled_sigma = sigma_median * best_factor
sub["Confidence"] = np.maximum(scaled_sigma, 70)




In [7]:
submission = sub[["Patient_Week", "FVC", "Confidence"]].copy()
submission.to_csv("submission.csv", index=False)

print("Submission file written to submission.csv")

Submission file written to submission.csv
